<div style="background-color:#EAF4EC; padding:20px; border-radius:10px;">

<h2 style="color:#2F6F4E; text-align:center; margin-bottom:5px;">
Hyperparameter Optimisation
</h2>

<h4 style="color:#2F6F4E; text-align:center; margin-top:0;">
Master Thesis – ESG Governance Indicators (EU-27)
</h4>

<p style="font-size:14px; color:#2F6F4E;">
This notebook implements the <strong>Hyperparameter Tuning</strong> stage of the CRISP-ML(Q) methodology.
The objective is to optimise the configurations of <strong>Random Forest</strong> and <strong>XGBoost</strong>
using the Optuna framework with TPE sampling, minimising out-of-sample RMSE on the validation set.
</p>

<p style="font-size:14px; color:#2F6F4E;">
⚠️ This notebook also evaluates <strong>ANN</strong> (MLPRegressor) and <strong>LSTM</strong> models as candidate approaches.
Both were subsequently excluded from the final pipeline due to inferior out-of-sample performance
relative to tree-based ensembles. See <code>08_model_selection.ipynb</code> for the full comparison.
</p>

</div>


In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks
import optuna


/opt/anaconda3/envs/ML/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
DATA_PATH = Path("../data/processed/data_final_governance_EU27.csv")
df_final = pd.read_csv(DATA_PATH)
print(df_final.shape)
df_final.head(1)

(405, 28)


,Country Name,Country Code,Indicator Name,Indicator Code,2000,2001,2002,2003,2004,2005,...,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023
0,Austria,AUT,Control of Corruption: Estimate,CC.EST,1.751059,1.751059,1.915159,1.963869,2.026624,1.912423,...,1.466502,1.472044,1.496881,1.50082,1.568587,1.521324,1.477709,1.242727,1.258587,1.133653


<div style="background-color:#EAF4EC; padding:16px; border-radius:10px;">
<h2 style="color:#2F6F4E; margin-bottom:5px; font-size:20px;">
Identification of Temporal Dimensions
</h2>
<p style="color:#2F6F4E; font-size:14px;">
Detection of year-based columns to structure the dataset in a time-aware format suitable for forecasting.
</p>
</div>


In [3]:
# Identify year columns (e.g. 2000, 2001, ..., 2023)
year_cols = [c for c in df_final.columns if c.isdigit()]

print(f"Detected {len(year_cols)} year columns:")
print(year_cols[:5], "...", year_cols[-5:])

Detected 24 year columns:
['2000', '2001', '2002', '2003', '2004'] ... ['2019', '2020', '2021', '2022', '2023']


<div style="background-color:#EAF4EC; padding:16px; border-radius:10px;">
<h2 style="color:#2F6F4E; margin-bottom:5px; font-size:20px;">
Reshaping Data to Long Format
</h2>
<p style="color:#2F6F4E; font-size:14px;">
Conversion from wide to long format to obtain a country–indicator–year structure required for time series modeling.
</p>
</div>


In [4]:
# Melt to long format (country–year–indicator)
df_long = df_final.melt(
    id_vars=["Country Name", "Country Code", "Indicator Name", "Indicator Code"],
    value_vars=year_cols,
    var_name="year",
    value_name="value"
)

df_long["year"] = df_long["year"].astype(int)

print(df_long.shape)
df_long.head(1)

(9720, 6)


,Country Name,Country Code,Indicator Name,Indicator Code,year,value
0,Austria,AUT,Control of Corruption: Estimate,CC.EST,2000,1.751059


Each line represetn 1 country x 1 indicator x 1 year

<div style="background-color:#EAF4EC; padding:16px; border-radius:10px;">
<h2 style="color:#2F6F4E; margin-bottom:5px; font-size:20px;">
Indicator Selection for Forecasting
</h2>
<p style="color:#2F6F4E; font-size:14px;">
Isolation of a single governance indicator to enable focused and interpretable forecasting experiments.
</p>
</div>


In [5]:
# Select one indicator to start with
indicator_name = "Control of Corruption: Estimate"

df_ind = df_long[df_long["Indicator Name"] == indicator_name].copy()

print(df_ind.shape)
df_ind.head(1)

(648, 6)


,Country Name,Country Code,Indicator Name,Indicator Code,year,value
0,Austria,AUT,Control of Corruption: Estimate,CC.EST,2000,1.751059


In [6]:
df_ind["Country Name"].nunique(), df_ind["year"].min(), df_ind["year"].max()

(27, np.int64(2000), np.int64(2023))

<div style="background-color:#EAF4EC; padding:16px; border-radius:10px;">
<h2 style="color:#2F6F4E; margin-bottom:5px; font-size:20px;">
Lag Feature Engineering
</h2>
<p style="color:#2F6F4E; font-size:14px;">
Generation of lagged values to capture short-term temporal dependencies within each country.
</p>
</div>


In [7]:
# Create lag features (per country)
for lag in [1, 2, 3]:
    df_ind[f"lag_{lag}"] = (
        df_ind
        .groupby("Country Name")["value"]
        .shift(lag)
    )

df_ind.head(1)

,Country Name,Country Code,Indicator Name,Indicator Code,year,value,lag_1,lag_2,lag_3
0,Austria,AUT,Control of Corruption: Estimate,CC.EST,2000,1.751059,NaN,NaN,NaN


<div style="background-color:#EAF4EC; padding:16px; border-radius:10px;">
<h2 style="color:#2F6F4E; margin-bottom:5px; font-size:20px;">
Rolling Window Statistics
</h2>
<p style="color:#2F6F4E; font-size:14px;">
Computation of rolling mean and volatility measures to smooth noise and represent medium-term trends.
</p>
</div>


In [8]:
# Garantir ordenação temporal correta
df_ind = df_ind.sort_values(["Country Name", "year"]).copy()

g = df_ind.groupby("Country Name")["value"]

# Rolling statistics (window = 3 years) — apenas informação passada
df_ind["rolling_mean_3"] = g.transform(
    lambda s: s.shift(1).rolling(window=3).mean()
)

df_ind["rolling_std_3"] = g.transform(
    lambda s: s.shift(1).rolling(window=3).std()
)

# Trend (short-term momentum)
df_ind["trend"] = df_ind["lag_1"] - df_ind["lag_2"]

df_ind.head(1)

,Country Name,Country Code,Indicator Name,Indicator Code,year,value,lag_1,lag_2,lag_3,rolling_mean_3,rolling_std_3,trend
0,Austria,AUT,Control of Corruption: Estimate,CC.EST,2000,1.751059,NaN,NaN,NaN,NaN,NaN,NaN


<div style="background-color:#EAF4EC; padding:16px; border-radius:10px;">
<h2 style="color:#2F6F4E; margin-bottom:5px; font-size:20px;">
Handling Initial Missing Values
</h2>
<p style="font-size:14px; color:#2F6F4E">
Removal of initial observations affected by lag and rolling window construction to ensure model validity.
</p>
</div>


In [9]:
# Drop initial rows with NaN caused by lags/rolling
req = ["value","lag_1","lag_2","lag_3","rolling_mean_3","rolling_std_3","trend"]
df_model = df_ind.dropna(subset=req).reset_index(drop=True)

print(df_model.shape)
df_model.head(1)

(567, 12)


,Country Name,Country Code,Indicator Name,Indicator Code,year,value,lag_1,lag_2,lag_3,rolling_mean_3,rolling_std_3,trend
0,Austria,AUT,Control of Corruption: Estimate,CC.EST,2003,1.963869,1.915159,1.751059,1.751059,1.805759,0.094743,0.164101


Initial observations lacking sufficient historical context were removed after temporal feature construction to ensure model validity and prevent information leakage.

<div style="background-color:#EAF4EC; padding:16px; border-radius:10px;">
<h2 style="color:#2F6F4E; margin-bottom:5px; font-size:20px;">
Time-Based Dataset Splitting
</h2>
<p style="color:#2F6F4E; font-size:14px;">
Separation of data into training, validation, and test sets following a strict chronological order.
</p>
</div>


In [10]:
train_end_year = 2018
val_end_year = 2021

# Split using time (NO shuffle, NO random seed)
train_df = df_model[df_model["year"] <= train_end_year].copy()
val_df = df_model[(df_model["year"] > train_end_year) & (df_model["year"] <= val_end_year)].copy()
test_df = df_model[df_model["year"] > val_end_year].copy()

# Basic checks
print("Train:", train_df.shape, "| Years:", train_df["year"].min(), "-", train_df["year"].max())
print("Val:  ", val_df.shape,   "| Years:", val_df["year"].min(), "-", val_df["year"].max())
print("Test: ", test_df.shape,  "| Years:", test_df["year"].min(), "-", test_df["year"].max())

Train: (432, 12) | Years: 2003 - 2018
Val:   (81, 12) | Years: 2019 - 2021
Test:  (54, 12) | Years: 2022 - 2023


Percentage-based splits were avoided due to the temporal nature of the forecasting task.

<div style="background-color:#EAF4EC; padding:16px; border-radius:10px;">
<h2 style="color:#2F6F4E; margin-bottom:5px; font-size:20px;">
Evaluation Metrics
</h2>
<p style="color:#2F6F4E; font-size:14px;">
To assess forecasting performance, this study employs Mean Absolute Error (MAE) and Root Mean Squared Error (RMSE).
</p>
</div>


In [11]:
def eval_metrics(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    return mae, rmse

<div style="background-color:#EAF4EC; padding:16px; border-radius:10px;">
<h2 style="color:#2F6F4E; margin-bottom:5px; font-size:20px;">
Features and Target Definition
</h2>
</div>

In [20]:
features = ["lag_1","lag_2","lag_3","rolling_mean_3","rolling_std_3","trend"]
target = "value"

In [13]:
X_train = train_df[features]
y_train = train_df[target]

X_val = val_df[features]
y_val = val_df[target]

X_test = test_df[features]
y_test = test_df[target]

<div style="background-color:#EAF4EC; padding:16px; border-radius:10px;">
<h2 style="color:#2F6F4E; margin-bottom:5px; font-size:20px;">
Forecasting Models
</h2>
<p style="color:#2F6F4E; font-size:14px;">
Implementation of Random Forest and XGBoost.
</p>
</div>


<div style="background-color:#EAF4EC; padding:16px; border-radius:10px;">
<h2 style="color:#2F6F4E; margin-bottom:5px; font-size:20px;">
Random Forest
</h2>
</div>

In [14]:
rf = RandomForestRegressor(
    n_estimators=400,
    max_depth=6,
    min_samples_split=8,
    min_samples_leaf=4,
    max_features="sqrt",
    bootstrap=True,
    random_state=42,
    n_jobs=-1)

rf.fit(X_train, y_train)

val_pred = rf.predict(X_val)
test_pred = rf.predict(X_test)

mae_val, rmse_val = eval_metrics(y_val, val_pred)
mae_test, rmse_test = eval_metrics(y_test, test_pred)

print("Random Forest")
print(f"Validation -> MAE: {mae_val:.4f} | RMSE: {rmse_val:.4f}")
print(f"Test       -> MAE: {mae_test:.4f} | RMSE: {rmse_test:.4f}")

Random Forest
Validation -> MAE: 0.0922 | RMSE: 0.1234
Test       -> MAE: 0.0615 | RMSE: 0.0774


<div style="background-color:#EAF4EC; padding:16px; border-radius:10px;">
<h2 style="color:#2F6F4E; margin-bottom:5px; font-size:20px;">
XGBoost 
</h2>
</div>

In [15]:
xgb = XGBRegressor(
    n_estimators=800,
    learning_rate=0.03,
    max_depth=3,
    min_child_weight=5,
    subsample=0.8,
    colsample_bytree=0.9,
    reg_alpha=0.05,
    reg_lambda=2.0,
    objective="reg:squarederror",
    random_state=42,
    n_jobs=-1
)

# Fit with validation monitoring (no leakage; uses X_val/y_val)
xgb.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    verbose=False
)

val_pred = xgb.predict(X_val)
test_pred = xgb.predict(X_test)

mae_val, rmse_val = eval_metrics(y_val, val_pred)
mae_test, rmse_test = eval_metrics(y_test, test_pred)

print("XGBoost")
print(f"Validation -> MAE: {mae_val:.4f} | RMSE: {rmse_val:.4f}")
print(f"Test       -> MAE: {mae_test:.4f} | RMSE: {rmse_test:.4f}")

XGBoost
Validation -> MAE: 0.0866 | RMSE: 0.1178
Test       -> MAE: 0.0694 | RMSE: 0.0847


<div style="background-color:#EAF4EC; padding:16px; border-radius:10px;">
<h2 style="color:#2F6F4E; margin-bottom:5px; font-size:20px;">
Hyperparamethrs Optimization using Optuna
</h2>
</div>

<div style="background-color:#EAF4EC; padding:16px; border-radius:10px;">
<h2 style="color:#2F6F4E; margin-bottom:5px; font-size:20px;">
XGBoost
</h2>
</div>

In [16]:
def objective_xgb(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 200, 3000),
        "learning_rate": trial.suggest_float("learning_rate", 0.005, 0.2, log=True),
        "max_depth": trial.suggest_int("max_depth", 2, 8),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 1.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 20.0, log=True),
        "objective": "reg:squarederror",
        "random_state": 42,
        "n_jobs": -1,
    }

    model = XGBRegressor(**params)
    model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)

    val_pred = model.predict(X_val)
    _, rmse_val = eval_metrics(y_val, val_pred)

    return rmse_val

study_xgb = optuna.create_study(direction="minimize", study_name="XGB_CC_EST", sampler=optuna.samplers.TPESampler(seed=42))
study_xgb.optimize(objective_xgb, n_trials=80)  # ajusta (ex: 50–150)

print("Best XGB RMSE (val):", study_xgb.best_value)
print("Best XGB params:", study_xgb.best_params)

[I 2026-02-02 22:16:19,021] A new study created in memory with name: XGB_CC_EST
[I 2026-02-02 22:16:20,359] Trial 0 finished with value: 0.12871362918059467 and parameters: {'n_estimators': 1494, 'learning_rate': 0.04610971287273479, 'max_depth': 7, 'min_child_weight': 3, 'subsample': 0.9930020025919482, 'colsample_bytree': 0.9742058598283471, 'reg_alpha': 0.0014673245997617091, 'reg_lambda': 0.25496981064594754}. Best is trial 0 with value: 0.12871362918059467.
[I 2026-02-02 22:16:22,116] Trial 1 finished with value: 0.12247425087731365 and parameters: {'n_estimators': 1988, 'learning_rate': 0.029572136680707836, 'max_depth': 4, 'min_child_weight': 8, 'subsample': 0.8811753460288437, 'colsample_bytree': 0.9670618053288995, 'reg_alpha': 4.439437013489371e-07, 'reg_lambda': 0.39155185232485423}. Best is trial 1 with value: 0.12247425087731365.
[I 2026-02-02 22:16:22,948] Trial 2 finished with value: 0.1225322618102087 and parameters: {'n_estimators': 589, 'learning_rate': 0.064327730390

Best XGB RMSE (val): 0.1140430916791721
Best XGB params: {'n_estimators': 1502, 'learning_rate': 0.01406430106815356, 'max_depth': 2, 'min_child_weight': 8, 'subsample': 0.6509501606637498, 'colsample_bytree': 0.6795541596326934, 'reg_alpha': 0.0004928380438312149, 'reg_lambda': 0.210226460490979}


In [17]:
best_xgb = XGBRegressor(
    **study_xgb.best_params,
    objective="reg:squarederror",
    random_state=42,
    n_jobs=-1
)

best_xgb.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)

val_pred = best_xgb.predict(X_val)
test_pred = best_xgb.predict(X_test)

mae_val, rmse_val = eval_metrics(y_val, val_pred)
mae_test, rmse_test = eval_metrics(y_test, test_pred)

print("\nXGBoost (Optuna Best)")
print(f"Validation -> MAE: {mae_val:.4f} | RMSE: {rmse_val:.4f}")
print(f"Test       -> MAE: {mae_test:.4f} | RMSE: {rmse_test:.4f}")



XGBoost (Optuna Best)
Validation -> MAE: 0.0845 | RMSE: 0.1140
Test       -> MAE: 0.0648 | RMSE: 0.0796


<div style="background-color:#EAF4EC; padding:16px; border-radius:10px;">
<h2 style="color:#2F6F4E; margin-bottom:5px; font-size:20px;">
Random Forest
</h2>
</div>

In [18]:
def objective_rf(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 200, 1200),
        "max_depth": trial.suggest_int("max_depth", 3, 20),
        "min_samples_split": trial.suggest_int("min_samples_split", 2, 20),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 10),
        "max_features": trial.suggest_float("max_features", 0.4, 1.0),
        "bootstrap": trial.suggest_categorical("bootstrap", [True, False]),
        "random_state": 42,
        "n_jobs": -1,
    }

    model = RandomForestRegressor(**params)
    model.fit(X_train, y_train)

    val_pred = model.predict(X_val)
    _, rmse_val = eval_metrics(y_val, val_pred)

    return rmse_val

study_rf  = optuna.create_study(direction="minimize", study_name="RF_CC_EST",  sampler=optuna.samplers.TPESampler(seed=42))
study_rf.optimize(objective_rf, n_trials=80)

print("Best RF RMSE (val):", study_rf.best_value)
print("Best RF params:", study_rf.best_params)


[I 2026-02-02 22:17:35,708] A new study created in memory with name: RF_CC_EST
[I 2026-02-02 22:17:36,092] Trial 0 finished with value: 0.11941102629985967 and parameters: {'n_estimators': 1057, 'max_depth': 15, 'min_samples_split': 11, 'min_samples_leaf': 10, 'max_features': 0.880197402689122, 'bootstrap': False}. Best is trial 0 with value: 0.11941102629985967.
[I 2026-02-02 22:17:36,452] Trial 1 finished with value: 0.12051293964206435 and parameters: {'n_estimators': 876, 'max_depth': 13, 'min_samples_split': 11, 'min_samples_leaf': 1, 'max_features': 0.8510097801575387, 'bootstrap': False}. Best is trial 0 with value: 0.11941102629985967.
[I 2026-02-02 22:17:36,907] Trial 2 finished with value: 0.1191953086924363 and parameters: {'n_estimators': 960, 'max_depth': 11, 'min_samples_split': 12, 'min_samples_leaf': 9, 'max_features': 0.8182624239866274, 'bootstrap': True}. Best is trial 2 with value: 0.1191953086924363.
[I 2026-02-02 22:17:37,211] Trial 3 finished with value: 0.119080

Best RF RMSE (val): 0.11604047428852934
Best RF params: {'n_estimators': 538, 'max_depth': 5, 'min_samples_split': 6, 'min_samples_leaf': 4, 'max_features': 0.8540989975527579, 'bootstrap': True}


In [19]:
best_rf = RandomForestRegressor(
    **study_rf.best_params,
    random_state=42,
    n_jobs=-1
)

best_rf.fit(X_train, y_train)

val_pred = best_rf.predict(X_val)
test_pred = best_rf.predict(X_test)

mae_val, rmse_val = eval_metrics(y_val, val_pred)
mae_test, rmse_test = eval_metrics(y_test, test_pred)

print("\nRandom Forest (Optuna Best)")
print(f"Validation -> MAE: {mae_val:.4f} | RMSE: {rmse_val:.4f}")
print(f"Test       -> MAE: {mae_test:.4f} | RMSE: {rmse_test:.4f}")


Random Forest (Optuna Best)
Validation -> MAE: 0.0836 | RMSE: 0.1160
Test       -> MAE: 0.0604 | RMSE: 0.0735
